Data Ingestion


In [1]:
from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="LangChain is a framework for building LLM applications.",
    metadata={
        "source": "notes.txt",
        "page": 1
    }
)
doc

Document(metadata={'source': 'notes.txt', 'page': 1}, page_content='LangChain is a framework for building LLM applications.')

In [3]:
## creating a txtfile
import os
os.makedirs("../data/text_files",exist_ok=True)

saving some texts as sample 

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


Using textloaders to read the text

In [6]:
##Text loader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt",encoding = "utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False

)

documents=dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

Rag Pipline

In [11]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   # <-- changed this line
from pathlib import Path

In [12]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: Deepak sharma.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-16T13:51:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-16T13:51:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Deepak sharma.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Deepak sharma.pdf', 'file_type': 'pdf'}, page_content='Deepak Sharma\ndeepakshar661@gmail.com|+91 8395918870|linkedin.com/in/deepak-sharma-aa22892b7|\ngithub.com/deepak-sh-07\nCareer Objective\nFull-Stack Developer with hands-on experience building scalable web applications, real-time systems, and\nRESTful APIs using the Node.js/Express ecosystem. Strong foundation in backend development, database\ndesign, and clean, maintainable code.\nTechnical Skills\nLanguages:JavaScript(ES6+), C++, SQL\nF rontend:React.js, Next.js, HTML5, 

In [14]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [15]:
chunks = split_documents(all_pdf_documents)
chunks

Split 1 documents into 4 chunks

Example chunk:
Content: Deepak Sharma
deepakshar661@gmail.com|+91 8395918870|linkedin.com/in/deepak-sharma-aa22892b7|
github.com/deepak-sh-07
Career Objective
Full-Stack Developer with hands-on experience building scalable w...
Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-16T13:51:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-16T13:51:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Deepak sharma.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Deepak sharma.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-07-16T13:51:12+00:00', 'author': '', 'keywords': '', 'moddate': '2026-07-16T13:51:12+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Deepak sharma.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Deepak sharma.pdf', 'file_type': 'pdf'}, page_content='Deepak Sharma\ndeepakshar661@gmail.com|+91 8395918870|linkedin.com/in/deepak-sharma-aa22892b7|\ngithub.com/deepak-sh-07\nCareer Objective\nFull-Stack Developer with hands-on experience building scalable web applications, real-time systems, and\nRESTful APIs using the Node.js/Express ecosystem. Strong foundation in backend development, database\ndesign, and clean, maintainable code.\nTechnical Skills\nLanguages:JavaScript(ES6+), C++, SQL\nF rontend:React.js, Next.js, HTML5, 

Embeddings and vectorStoreDB

In [16]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\Deepak\Code\python\Code\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
